# Readme

# imports

In [ ]:
import photoproductionmodel as pm
import branching_ratios as br
import numpy as np
import scipy
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
from google.colab import output
output.enable_custom_widget_manager()


mp = 0.9382720813
e = 0.303
# gauge coupling charges
gauge_couplings = {"A'":np.array([2/3,-1/3,-1/3,-1,-1,0,0,0]),
                   "B-L":np.array([1/3,1/3,1/3,-1,-1,-1,-1,-1]),
                   "B":np.array([1/3,1/3,1/3,-e**2/(4*np.pi)**2,-e**2/(4*np.pi)**2,0,0,0]),
                   "Chargephobic":np.array([-1/(3*np.sqrt(2)),np.sqrt(2)/3,np.sqrt(2)/3,0,0,-1/np.sqrt(2),-1/np.sqrt(2),-1/np.sqrt(2)])}


#photoproduction model params CONVERGED
params = np.array([11.81336049,
                   3.92112205,
                   0.77877409,
                   0.68472406,
                   0.89693757,
                   0.80841249,
                   0.61509084,
                   0.69624785,
                   -4.21703614,
                   1.90289369,
                   8.81722149,
                   3.12168375,
                   1.1120275,
                   9.47975335,
                   0.92746299,
                   0.80066329,
                   0.80370305,
                   0.56739554,
                   1.74087675,
                   2.21138913,
                   5.52013592,
                   1.06419093,
                   0.99383077])

## Rotate 3d plots install (optional)

In [ ]:
# !pip install ipympl
'''
1. run the install above
2. restart kernel
3. do not run the install again
4. run %matplotlib widget
5. continue
'''

In [ ]:
# %matplotlib widget

# Acceptance function

In [ ]:
def A(cosX,EX,L0,l0,alphaX,mX,x):
  '''
  returns
    A(cosX,EX) the acceptance fraction or probability of decaying after shield and before detector
  L0
    float distance from production point to end of shield
  l0
    float distance from production point to detector
  alphaX
    float coupling constant
  mX
    float mass of X boson
  x
    ndarray of floats gauge couplings [u,d,s,e,mu,nu_e,nu_mu,nu_tau]
  '''

  if cosX < 0:
    return np.nan

  gamma = EX/mX
  c_Tau = br.get_decay_length(mX,alphaX,x) # femtometers

  return np.exp(-L0/(gamma*c_Tau*cosX)) - np.exp(-l0/(gamma*c_Tau*cosX))

# Integrating total cross section and acceptance

### submethods

In [ ]:
def kinematically_allowed_region_EX_cosX(s_tot,mX,resolution):
  '''
  s_tot
   float experimental c.o.m. energy squared
  mX
   float mass of X boson
  resolution
   int resolution of rectangular point cloud
  return
   EX_allowed, cosX_allowed numpy arrays of allowed coordinates in (EX,cosX)
  '''
  E_beam = (s_tot - mp**2) / (2 * mp)

  # uniform point cloud in rectangular region
  cosX = np.linspace(-1,1,resolution)
  EX = np.linspace(mX + 1e-6,E_beam,resolution)

  EX_grid, cosX_grid = np.meshgrid(EX,cosX)

  # point cloud limited to kinematically allowed region (1D objs: EX_allowed, cosX_allowed)
  qX = np.sqrt(EX_grid**2 - mX**2)
  nu = (mp * EX_grid - 0.5 * mX**2) / (mp - EX_grid + qX * cosX_grid)

  validity_mask = (nu>EX) & (nu<E_beam)
  EX_allowed, cosX_allowed = EX_grid[validity_mask] , cosX_grid[validity_mask]

  # redoing the above steps for strictly defined resolution in boundary given above
  # I want cosX spacing around 0.002, and EX spacing around 0.04, for all s_tot and mX
  target_dcos = 0.002
  target_dEX = 0.04

  cosX_allowed_min = np.min(cosX_allowed)
  cosX_allowed_max = np.max(cosX_allowed)
  N_cosX = int(np.ceil((cosX_allowed_max-cosX_allowed_min)/target_dcos)) + 1

  EX_allowed_min = np.min(EX_allowed)
  EX_allowed_max = np.max(EX_allowed)
  N_EX = int(np.ceil((EX_allowed_max-EX_allowed_min)/target_dEX)) + 1

  cosX2 = np.linspace(cosX_allowed_min,cosX_allowed_max,N_cosX)
  EX2 = np.linspace(EX_allowed_min,EX_allowed_max,N_EX)

  EX_grid2, cosX_grid2 = np.meshgrid(EX2,cosX2)

  qX2 = np.sqrt(EX_grid2**2 - mX**2)
  nu2 = (mp * EX_grid2 - 0.5 * mX**2) / (mp - EX_grid2 + qX2 * cosX_grid2)

  validity_mask2 = (nu2>EX2) & (nu2<E_beam)
  EX_allowed2, cosX_allowed2 = EX_grid2[validity_mask2] , cosX_grid2[validity_mask2]

  return EX_allowed2, cosX_allowed2

### plotting regions and integrands

In [ ]:
def plot_dsigX_dEX_dcosX(s_tot,params,mX,xq,resolution):
  '''
  3d point cloud of dsigX_dEX_dcosX
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  dsig = np.zeros(len(EX_allowed))
  for i in range(len(EX_allowed)):
    dsig[i] = pm.dsigX_dEX_dcosX(s_tot,EX_allowed[i],cosX_allowed[i],params,mX,xq)

  # plotting point cloud
  fig = plt.figure()
  point_cloud_ax = plt.axes(projection='3d')
  point_cloud_ax.scatter(EX_allowed, cosX_allowed, dsig,s=0.01)
  point_cloud_ax.set_xlabel('EX')
  point_cloud_ax.set_ylabel('cosX')
  point_cloud_ax.set_zlabel('dsigX_dEX_dcosX')
  point_cloud_ax.set_title('dsigX_dEX_dcosX')
  plt.show()

In [ ]:
def plot_A(L0,l0,alphaX,mX,x):
  '''
  plots 3d point cloud over allowed region of (EX,cosX) of A(EX,cosX)

  L0
    float shield length
  alphaX
    float coupling constant
  mX
    float mass of X boson
  x
    ndarray of floats gauge couplings [u,d,s,e,mu,nu_e,nu_mu,nu_tau]
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  A_vals = np.zeros(len(EX_allowed))
  for i in range(len(EX_allowed)):
    A_vals[i] = A(cosX_allowed[i],EX_allowed[i],L0,l0,alphaX,mX,x)

  fig = plt.figure()
  point_cloud_ax = plt.axes(projection='3d')
  point_cloud_ax.scatter(EX_allowed, cosX_allowed, A_vals,marker='v',s=0.01)
  point_cloud_ax.set_xlabel('EX')
  point_cloud_ax.set_ylabel('cosX')
  point_cloud_ax.set_zlabel('A')
  point_cloud_ax.set_title('A')
  plt.show()

In [ ]:
def plot_A_times_dsigX_dEX_dcosX(s_tot,params,mX,xq,x,alphaX,L0,l0,resolution):
  '''
  3d point cloud of dsigX_dEX_dcosX * A(EX,cosX)
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  A_times_dsig = np.zeros(len(EX_allowed))
  for i in range(len(EX_allowed)):
    A_times_dsig[i] = pm.dsigX_dEX_dcosX(s_tot,EX_allowed[i],cosX_allowed[i],params,mX,xq) * A(cosX_allowed[i],EX_allowed[i],L0,l0,alphaX,mX,x)

  # plotting point cloud
  fig = plt.figure()
  point_cloud_ax = plt.axes(projection='3d')
  point_cloud_ax.scatter(EX_allowed, cosX_allowed, A_times_dsig,marker='v',s=0.01)
  point_cloud_ax.set_xlabel('EX')
  point_cloud_ax.set_ylabel('cosX')
  point_cloud_ax.set_zlabel('dsigX_dEX_dcosX')
  point_cloud_ax.set_title('A * dsigX_dEX_dcosX')
  plt.show()

In [ ]:
def plot_allowed_region(s_tot,mX,resolution):
  '''
  plots point cloud of allowed region
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  fig = plt.figure()
  plt.scatter(EX_allowed,cosX_allowed,s=0.1)
  plt.title('valid threshold')
  plt.xlabel('EX')
  plt.ylabel('cosX')

In [ ]:
def spline_region(s_tot,mX,resolution,plot = False):
  '''
  splines for dblquad integration
  plots point cloud allowed region and splines of upper and lower EX(cosX)
  return sequence of CubicSpline objects (spline_EX_upper, spline_EX_lower)
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  # interpolating region boundary functions of EX wrt cosX
  cosX_unique = np.unique(cosX_allowed)

  EX_upper = np.zeros(len(cosX_unique))
  EX_lower = np.zeros(len(cosX_unique)) # arrays of upper and lower values of EX for the set of unique cosXs in our point cloud

  for i in range(len(cosX_unique)):
    key = np.where(cosX_allowed == cosX_unique[i],True,False)
    EX_upper[i]= np.max(EX_allowed[key])
    EX_lower[i] = np.min(EX_allowed[key])

  spline_EX_upper = scipy.interpolate.CubicSpline(cosX_unique,EX_upper,extrapolate=False)
  spline_EX_lower = scipy.interpolate.CubicSpline(cosX_unique,EX_lower,extrapolate=False)

  if plot:
    fig = plt.figure()
    # plt.scatter(EX_upper,cosX_unique,s=0.1)
    cosX_range = np.linspace(np.min(cosX_unique),np.max(cosX_unique),resolution)
    plt.plot(spline_EX_upper(cosX_range),cosX_range,color='g')
    plt.plot(spline_EX_lower(cosX_range),cosX_range,color='r')
    plt.title('valid threshold, splines')
    plt.xlabel('EX')
    plt.ylabel('cosX')
    plt.show()

  return spline_EX_upper, spline_EX_lower

### computing integrals

In [ ]:
def dsig_integrand(EX,cosX,s_tot,params,mX,xq):
  '''
  integrand formatted for dblquad integrating
  '''
  return pm.dsigX_dEX_dcosX(s_tot,EX,cosX,params,mX,xq)

def dsig_times_acceptance_integrand(EX,cosX,s_tot,params,mX,xq,L0,l0,alphaX,x):
  '''
  integrand formatted for dblquad integrating
  '''
  return pm.dsigX_dEX_dcosX(s_tot,EX,cosX,params,mX,xq) * A(cosX,EX,L0,l0,alphaX,mX,x)

def integrate_dsig(s_tot,params,mX,xq,resolution):
  '''
  integrates d_sigX_dEX_dcosX with scipy.dblquad
  return float value of integral
  '''
  # region boundaries
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)
  cosX_lower_boundary = np.min(cosX_allowed)
  cosX_upper_boundary = np.max(cosX_allowed)
  EX_upper_boundary_func , EX_lower_boundary_func = spline_region(s_tot,mX,resolution)

  # integration
  result , error = scipy.integrate.dblquad(
    dsig_integrand,
    cosX_lower_boundary,
    cosX_upper_boundary,
    EX_lower_boundary_func,
    EX_upper_boundary_func,
    args=(s_tot,params,mX,xq)
    )

  return result

def integrate_dsig_times_acceptance(s_tot,params,mX,xq,L0,l0,alphaX,x,resolution):
  '''
  integrates d_sigX_dEX_dcosX * A(EX,cosX) with scipy.dblquad

  L0
    float shield length
  alphaX
    float coupling constant
  mX
    float mass of X boson
  x
    ndarray of floats gauge couplings [u,d,s,e,mu,nu_e,nu_mu,nu_tau]
  return
    float value of integral
  '''
  # region boundaries
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)
  cosX_lower_boundary = np.min(cosX_allowed)
  cosX_upper_boundary = np.max(cosX_allowed)
  EX_upper_boundary_func , EX_lower_boundary_func = spline_region(s_tot,mX,resolution)

  # integration
  result , error = scipy.integrate.dblquad(
    dsig_times_acceptance_integrand,
    cosX_lower_boundary,
    cosX_upper_boundary,
    EX_lower_boundary_func,
    EX_upper_boundary_func,
    args=(s_tot,params,mX,xq,L0,l0,alphaX,x)
    )

  return result


# Notes

##### integration method

* Create a point cloud in a rectangular region in *(EX,cosX)* that supercedes the kinematically allowed region

* Remove all points that are not kinematically allowed

* Create splines for the upper and lower curves of *EX(cosX)*

* Pass the integrand, the splines of *EX(cosX)*, and the upper and lower boundaries of *cosX* into ***scipy.integrate.dblquad(func, a, b, gfun, hfun)*** to perform the integral

##### results

$\frac{dN}{dE_XdcosX} = BR_{X→F} \cdot \ell \cdot \frac{d^2σ}{dE_XdcosX} \cdot A(E_X,cosX)$


---


$N = BR_{X→F} \cdot \ell \cdot \sigma \cdot E[A] $



---
$E[A] = \frac{\int∫A(E_X,cosX) \frac{d^2σ}{dE_XdcosX} dE_XdcosX}{\int\int\frac{d^2σ}{dE_XdcosX}dE_XdcosX} $


# Updates

* Fixed boundary resolution failure at low masses by implementing adaptive refinement.
* Fixed acceptance function definition, now defines the desired probability
* For low masses (0.2) E_expected is coming out NaN

# Execute code

### set parameters

In [ ]:
s_tot = 76
L0 = 100 #fm
l0 = 1000 #fm
mX = 0.2
x = gauge_couplings["Chargephobic"]
xq = x[0:3]
alphaX = 1/136
resolution = 500

### everything

In [ ]:
sig_tot = integrate_dsig(s_tot,params,mX,xq,resolution)
print('sig_tot = ' + str(f"{sig_tot:.16e}"))


sig_times_A_integral = integrate_dsig_times_acceptance(s_tot,params,mX,xq,L0,l0,alphaX,x,resolution)
A_expected = sig_times_A_integral / sig_tot
print(' E[A] = ' + str(f"{A_expected:.16e}"))

plot_dsigX_dEX_dcosX(s_tot,params,mX,xq,resolution)

plot_A_times_dsigX_dEX_dcosX(s_tot,params,mX,xq,x,alphaX,L0,l0,resolution)

plot_A(L0,l0,alphaX,mX,x)

spline_region(s_tot,mX,resolution,True)